In [1]:
import os
import json
import argparse
import numpy as np
import pandas as pd
from PIL import Image
from scipy.fft import dct 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b1
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import entropy
import optuna

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
HYPERS = {
    'batch_size': 64,
    'lr': 5e-5,
    'epochs': 20,
    'dropout': 0.2,
    'val_split': 0.15
}

In [4]:
class CIFakeDataset(Dataset):
    def __init__(self, transform=None, split='train'):
        self.root = "/kaggle/input/deepfake-ml-challenge/DATASET"
        self.split = split
        self.transform = transform

        self.images, self.labels = [], []

        self._load_split('real_cifake_images', 'real_cifake_preds.json')
        self._load_split('fake_cifake_images', 'fake_cifake_preds.json')

        total_size = len(self.images)
        split_idx = int(0.85 * total_size)
        if split == 'train':
            self.images = self.images[:split_idx]
            self.labels = self.labels[:split_idx]
        else:
            self.images = self.images[split_idx:]
            self.labels = self.labels[split_idx:]

    def _load_split(self, folder_name, json_name):
        folder = os.path.join(self.root, folder_name)
        json_path = os.path.join(self.root, json_name)

        with open(json_path, 'r') as f:
            data = json.load(f)

        for entry in data:
            img_idx = entry["index"]
            pred = entry.get("prediction", "").lower()
            img_path = os.path.join(folder, f"{img_idx}.png")

            if not os.path.exists(img_path):
                continue

            # Skip corrupt images
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except Exception:
                continue

            if pred == "real":
                label = 0
            elif pred == "fake":
                label = 1
            else:
                continue

            self.images.append(img_path)
            self.labels.append(label)



    def __len__(self):
        return len(self.images)

    def _compute_dct_features(self, image, block_size=16):
        gray = np.array(image.convert("L")) / 255.0
        h, w = gray.shape
        dct_full = dct(dct(gray.T, norm='ortho').T, norm='ortho')
        dct_block = dct_full[:block_size, :block_size].flatten()
        dct_vec = dct_block[:256] if len(dct_block) >= 256 else np.pad(dct_block, (0, 256 - len(dct_block)))
        return torch.tensor(dct_vec, dtype=torch.float32)
    

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert("RGB")
            dct = self._compute_dct_features(image)
        except Exception:
            image = Image.new("RGB", (224, 224), color=(255, 0, 0))

        if self.transform:
            image = self.transform(image)

        return {
            "image": image,
            "dct": dct,
            "label": torch.tensor(label, dtype=torch.long),
            "path": img_path
        }
        

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.1)], p=0.5),
    transforms.RandomAffine(15, shear=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [6]:
train_ds = CIFakeDataset(transform=train_transform, split='train')
val_ds = CIFakeDataset(transform=test_transform, split='val')

In [7]:
train_loader = DataLoader(train_ds, batch_size=HYPERS['batch_size'], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=HYPERS['batch_size'])

In [8]:
try:
    for batch in train_loader:
        print(f"Image batch shape: {batch['image'].shape}")
        print(f"DCT batch shape: {batch['dct'].shape}")
        print(f"Label batch shape: {batch['label'].shape}")
        print(f"Example labels: {batch['label'][:10].tolist()}")
        break 
except Exception as e:
    print(" Error while loading batch:", e)

Image batch shape: torch.Size([64, 3, 256, 256])
DCT batch shape: torch.Size([64, 256])
Label batch shape: torch.Size([64])
Example labels: [0, 1, 1, 0, 0, 1, 0, 1, 1, 1]


In [9]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b1

class SpatialBranch(nn.Module):
    def __init__(self):
        super().__init__()
        # EfficientNet backbone for 256x256 input
        self.backbone = efficientnet_b1(weights="IMAGENET1K_V1")
        self.backbone.classifier = nn.Identity()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1280, 512)  # EfficientNet-B1 feature dim → 512 projection
    
    def forward(self, x):
        feats = self.backbone.features(x)    # [B, 1280, 8, 8]
        feats = self.pool(feats).flatten(1)  # [B, 1280]
        return self.fc(feats)                # [B, 512]

class FreqBranch(nn.Module):
    def __init__(self, input_dim=256):
        super().__init__()
        # input: [B, 256]
        # treat it as a 1D signal of length 256 with 1 channel
        self.conv1 = nn.Conv1d(1, 128, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv1d(64, 32, kernel_size=3, stride=2, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, 512)
        self.act = nn.GELU()
    
    def forward(self, x):
        x = x.unsqueeze(1)                   # [B, 1, 256]
        x = self.act(self.conv1(x))          # [B, 128, 128]
        x = self.act(self.conv2(x))          # [B, 64, 64]
        x = self.act(self.conv3(x))          # [B, 32, 32]
        x = self.pool(x).flatten(1)          # [B, 32]
        return self.fc(x)                    # [B, 512]

class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=512, heads=4, dropout=0.3):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
    
    def forward(self, spatial, freq):
        # [B, 512] → [B, 1, 512]
        spatial = spatial.unsqueeze(1)
        freq = freq.unsqueeze(1)
        attn_out, _ = self.attn(spatial, freq, freq)
        fused = self.norm(attn_out.squeeze(1) + spatial.squeeze(1))
        concat = torch.cat([fused, freq.squeeze(1)], dim=1)  # [B, 1024]
        return self.mlp(concat)  # [B, 512]

class BinaryHead(nn.Module):
    def __init__(self, dim=512, dropout=0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(dim, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.head(x)  # logits

class SpectralClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.spatial = SpatialBranch()
        self.freq = FreqBranch(input_dim=256)
        self.fusion = CrossAttentionFusion(dim=512, heads=4, dropout=dropout)
        self.head = BinaryHead(dim=512, dropout=dropout)
    
    def forward(self, img, dct):
        spat = self.spatial(img)     # [B, 512]
        freq = self.freq(dct)        # [B, 512]
        fused = self.fusion(spat, freq)  # [B, 512]
        pred = self.head(fused)      # [B, 1]
        return pred

In [10]:
model = SpectralClassifier().to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1_rwightman-bac287d4.pth
100%|██████████| 30.1M/30.1M [00:00<00:00, 200MB/s]


In [11]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                                            Param #
SpectralClassifier                                                --
├─SpatialBranch: 1-1                                              --
│    └─EfficientNet: 2-1                                          --
│    │    └─Sequential: 3-1                                       6,513,184
│    │    └─AdaptiveAvgPool2d: 3-2                                --
│    │    └─Identity: 3-3                                         --
│    └─AdaptiveAvgPool2d: 2-2                                     --
│    └─Linear: 2-3                                                655,872
├─FreqBranch: 1-2                                                 --
│    └─Conv1d: 2-4                                                512
│    └─Conv1d: 2-5                                                24,640
│    └─Conv1d: 2-6                                                6,176
│    └─AdaptiveAvgPool1d: 2-7                                     --
│    └─Li

In [12]:
from tqdm import tqdm

def train_model(model, train_loader, val_loader, hypers):
    optimizer = optim.AdamW(model.parameters(), lr=hypers['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.BCEWithLogitsLoss()


    best_val_acc = 0.0

    for epoch in range(hypers['epochs']):
        model.train()
        total_train_loss, correct_train, total_train = 0.0, 0, 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{hypers['epochs']} [Train]"):
            imgs = batch['image'].to(device)
            dcts = batch['dct'].to(device)
            labels = batch['label'].float().to(device)

            optimizer.zero_grad()
            preds = model(imgs, dcts).squeeze(-1)

            loss = criterion(preds, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            predicted = (preds > 0.5).float()
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

        train_loss = total_train_loss / len(train_loader)
        train_acc = correct_train / total_train

        # -------- Validation Phase --------
        model.eval()
        total_val_loss, correct_val, total_val = 0.0, 0, 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{hypers['epochs']} [Val]"):
                imgs = batch['image'].to(device)
                dcts = batch['dct'].to(device)
                labels = batch['label'].float().to(device)

                preds = model(imgs, dcts).squeeze(-1)
                loss = criterion(preds, labels)

                total_val_loss += loss.item()
                predicted = (preds > 0.5).float()
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)

        val_loss = total_val_loss / len(val_loader)
        val_acc = correct_val / total_val

        scheduler.step(val_loss)


        print(f"Epoch [{epoch+1}/{hypers['epochs']}] "
              f"| Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% "
              f"| Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")

    return val_acc

In [13]:
def objective(trial):
    # Define hyperparameters to search
    HYPERS = {
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'lr': trial.suggest_loguniform('lr', 1e-5, 1e-3),
        'dropout': trial.suggest_float('dropout', 0.1, 0.5),
        'epochs': 15,
        'val_split': 0.15
    }

    # Build model with trial-specific dropout
    model = SpectralClassifier().to(device)
    model.fusion.mlp[2].p = HYPERS['dropout']
    for layer in model.head.head:
        if isinstance(layer, nn.Dropout):
            layer.p = HYPERS['dropout']

    val_acc = train_model(model, train_loader, val_loader, HYPERS)

    global best_val_acc, best_model_state
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict()

    return val_acc

    

In [14]:
best_val_acc = 0
best_model_state = None

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

[I 2025-11-01 23:07:38,441] A new study created in memory with name: no-name-d60bc86b-4ebd-46f4-8f72-d6ba00137eda
/tmp/ipykernel_37/3145561352.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'lr': trial.suggest_loguniform('lr', 1e-5, 1e-3),
Epoch 1/15 [Val]: 100%|██████████| 5/5 [00:01<00:00,  3.70it/s]


Epoch [1/15] | Train Loss: 0.5070, Train Acc: 74.47% | Val Loss: 0.3328, Val Acc: 85.00%


Epoch 2/15 [Val]: 100%|██████████| 5/5 [00:01<00:00,  3.83it/s]


Epoch [2/15] | Train Loss: 0.3163, Train Acc: 88.41% | Val Loss: 0.3036, Val Acc: 82.33%


Epoch 3/15 [Val]: 100%|██████████| 5/5 [00:01<00:00,  4.08it/s]


...

Epoch [13/15] | Train Loss: 0.1816, Train Acc: 94.12% | Val Loss: 0.2425, Val Acc: 87.67%


Epoch 14/15 [Val]: 100%|██████████| 5/5 [00:01<00:00,  3.97it/s]


Epoch [14/15] | Train Loss: 0.1611, Train Acc: 94.53% | Val Loss: 0.2560, Val Acc: 86.67%


Epoch 15/15 [Val]: 100%|██████████| 5/5 [00:01<00:00,  4.06it/s]
[I 2025-11-02 00:00:36,664] Trial 9 finished with value: 0.8766666666666667 and parameters: {'batch_size': 32, 'lr': 3.9474999204819066e-05, 'dropout': 0.12926731525294788}. Best is trial 1 with value: 0.92.


Epoch [15/15] | Train Loss: 0.1623, Train Acc: 94.59% | Val Loss: 0.2321, Val Acc: 87.67%


In [15]:
print("Best Hyperparameters:", study.best_params)
print("Best Validation Accuracy:", best_val_acc)

torch.save(best_model_state, "deepfake_model.pt")

Best Hyperparameters: {'batch_size': 32, 'lr': 0.0005547743057926064, 'dropout': 0.2732799212756177}
Best Validation Accuracy: 0.92
